# Optuna Hyperparameter Tuning

**Baseline**: MAE=850,249 | RMSE=1,180,664 | R²=0.8170  
**Objective**: Minimize OOF RMSE (đúng metric competition)  
**Strategy**: Tune LGBM và XGB riêng → tìm optimal blend weight → so sánh với baseline

**Ước tính thời gian**:
- 50 trials LGBM × 5 folds ≈ 15–25 phút (Kaggle P100)
- 30 trials XGB  × 5 folds ≈ 10–15 phút
- Tổng: ~35–40 phút

## 1 — Imports & load data

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import optuna
import json
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)  # chỉ print kết quả quan trọng

from scipy.optimize import minimize_scalar
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Paths
DATA_DIR  = '/kaggle/input/notebooks/harveycorn/features-engineering/'
WORK_DIR  = '/kaggle/working/'

TRAIN_FILE  = DATA_DIR + 'features_train_v2.csv'
PRED_FILE   = DATA_DIR + 'features_predict_v2.csv'
META_FILE   = DATA_DIR + 'feature_meta_v2.json'
OUTPUT_FILE = WORK_DIR + 'submission_optuna.csv'

TARGET      = 'Revenue'
DATE_COL    = 'Date'
RANDOM_SEED = 42

# CV config — giống hệt ensemble notebook
N_FOLDS        = 5
GAP_DAYS       = 90
VAL_DAYS       = 182
MIN_TRAIN_DAYS = 730

# Optuna config
N_TRIALS_LGBM = 50
N_TRIALS_XGB  = 30

print('✅ Imports OK')

In [ ]:
with open(META_FILE) as f:
    meta = json.load(f)

FEATURE_COLS = meta['feature_cols']
COGS_RATIO   = meta['cogs_ratio']

train_df = pd.read_csv(TRAIN_FILE, parse_dates=[DATE_COL])
pred_df  = pd.read_csv(PRED_FILE,  parse_dates=[DATE_COL])
train_df = train_df.sort_values(DATE_COL).reset_index(drop=True)
pred_df  = pred_df.sort_values(DATE_COL).reset_index(drop=True)
train_df['log_Revenue'] = np.log1p(train_df[TARGET])

# Chỉ dùng features có trong cả 2 df
FEATURE_COLS = [c for c in FEATURE_COLS if c in train_df.columns and c in pred_df.columns]

X = train_df[FEATURE_COLS].values
y = train_df['log_Revenue'].values

print(f'Train: {len(train_df)} rows | Features: {len(FEATURE_COLS)}')

## 2 — CV folds & metrics

In [ ]:
def make_folds(df, date_col, n_folds, gap_days, val_days, min_train_days):
    df = df.sort_values(date_col).reset_index(drop=True)
    total = len(df)
    step  = max(1, (total - val_days - gap_days - min_train_days) // n_folds)
    folds = []
    for i in range(n_folds):
        te = min_train_days + i * step
        vs = te + gap_days
        ve = vs + val_days
        if ve > total:
            break
        folds.append((list(range(0, te)), list(range(vs, ve))))
    return folds

folds = make_folds(train_df, DATE_COL, N_FOLDS, GAP_DAYS, VAL_DAYS, MIN_TRAIN_DAYS)

def oof_rmse(params, model_type='lgbm', early_stopping=100):
    """
    Chạy full CV với params cho trước, trả về OOF RMSE trên Revenue scale.
    Đây là objective function cho Optuna.
    """
    oof_preds  = np.full(len(X), np.nan)

    for train_idx, val_idx in folds:
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_vl, y_vl = X[val_idx],   y[val_idx]

        if model_type == 'lgbm':
            dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
            dval   = lgb.Dataset(X_vl, label=y_vl, reference=dtrain, free_raw_data=False)
            model  = lgb.train(
                params, dtrain,
                num_boost_round = params.get('n_estimators', 2000),
                valid_sets      = [dval],
                callbacks       = [lgb.early_stopping(early_stopping, verbose=False),
                                   lgb.log_evaluation(-1)]
            )
            log_pred = model.predict(X_vl, num_iteration=model.best_iteration)

        elif model_type == 'xgb':
            model = xgb.XGBRegressor(
                **params,
                early_stopping_rounds=early_stopping
            )
            model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
            log_pred = model.predict(X_vl)

        oof_preds[val_idx] = np.expm1(log_pred)

    mask    = ~np.isnan(oof_preds)
    actuals = train_df.loc[mask, TARGET].values
    preds   = np.clip(oof_preds[mask], 0, None)
    return np.sqrt(mean_squared_error(actuals, preds))

print(f'✅ {len(folds)} folds | Objective: OOF RMSE on Revenue scale')

## 3 — Tune LightGBM

In [ ]:
def lgbm_objective(trial):
    params = {
        'objective'        : 'regression',
        'metric'           : 'rmse',
        'boosting_type'    : 'gbdt',
        'verbose'          : -1,
        'random_state'     : RANDOM_SEED,
        'n_jobs'           : -1,
        'n_estimators'     : 2000,

        # ── Search space
        # num_leaves: complexity chính — tăng → fit tốt hơn nhưng dễ overfit
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 255),

        # min_child_samples: tối thiểu samples/leaf — tăng → regularize mạnh hơn
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),

        # learning_rate: nhỏ hơn → cần nhiều trees hơn nhưng generalize tốt hơn
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),

        # subsample + colsample: bagging — giảm variance
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq'   : 1,
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),

        # regularization
        'reg_alpha'        : trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.0, 5.0),

        # min_split_gain: threshold để split — tăng → pruning mạnh hơn
        'min_split_gain'   : trial.suggest_float('min_split_gain', 0.0, 1.0),
    }
    return oof_rmse(params, model_type='lgbm')


print(f'Tuning LightGBM — {N_TRIALS_LGBM} trials...')
print(f'Baseline OOF RMSE: 1,180,664')
print()

lgbm_study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.TPESampler(seed=RANDOM_SEED),
    pruner    = optuna.pruners.MedianPruner(n_warmup_steps=10)
)

# Seed với baseline params để có điểm khởi đầu tốt
lgbm_study.enqueue_trial({
    'num_leaves': 63, 'min_child_samples': 50, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'min_split_gain': 0.0
})

lgbm_study.optimize(
    lgbm_objective,
    n_trials  = N_TRIALS_LGBM,
    callbacks = [
        # Print progress mỗi 5 trials
        lambda study, trial: print(
            f'  Trial {trial.number:>3} | RMSE={trial.value:>12,.0f} | '
            f'Best={study.best_value:>12,.0f}'
        ) if trial.number % 5 == 0 else None
    ]
)

print()
print(f'✅ Best LGBM RMSE : {lgbm_study.best_value:,.0f}')
print(f'   Improvement    : {1_180_664 - lgbm_study.best_value:+,.0f}')
print(f'   Best params:')
for k, v in lgbm_study.best_params.items():
    print(f'     {k:<22}: {v}')

## 4 — Tune XGBoost

In [ ]:
def xgb_objective(trial):
    params = {
        'objective'        : 'reg:squarederror',
        'eval_metric'      : 'rmse',
        'booster'          : 'gbtree',
        'random_state'     : RANDOM_SEED,
        'n_jobs'           : -1,
        'verbosity'        : 0,
        'n_estimators'     : 2000,

        # ── Search space
        'max_depth'        : trial.suggest_int('max_depth', 3, 8),
        'min_child_weight' : trial.suggest_int('min_child_weight', 10, 100),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'alpha'            : trial.suggest_float('alpha', 0.0, 2.0),   # L1
        'lambda'           : trial.suggest_float('lambda', 0.0, 5.0),  # L2
        'gamma'            : trial.suggest_float('gamma', 0.0, 1.0),   # min split loss
    }
    return oof_rmse(params, model_type='xgb')


print(f'Tuning XGBoost — {N_TRIALS_XGB} trials...')
print()

xgb_study = optuna.create_study(
    direction = 'minimize',
    sampler   = optuna.samplers.TPESampler(seed=RANDOM_SEED),
)

# Seed baseline
xgb_study.enqueue_trial({
    'max_depth': 5, 'min_child_weight': 50, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'colsample_bylevel': 1.0,
    'alpha': 0.1, 'lambda': 1.0, 'gamma': 0.0
})

xgb_study.optimize(
    xgb_objective,
    n_trials  = N_TRIALS_XGB,
    callbacks = [
        lambda study, trial: print(
            f'  Trial {trial.number:>3} | RMSE={trial.value:>12,.0f} | '
            f'Best={study.best_value:>12,.0f}'
        ) if trial.number % 5 == 0 else None
    ]
)

print()
print(f'✅ Best XGB RMSE  : {xgb_study.best_value:,.0f}')
print(f'   Improvement   : {1_180_664 - xgb_study.best_value:+,.0f}')
print(f'   Best params:')
for k, v in xgb_study.best_params.items():
    print(f'     {k:<22}: {v}')

## 5 — Optuna visualization

In [ ]:
# ── Trial history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, study, name in [
    (axes[0], lgbm_study, 'LightGBM'),
    (axes[1], xgb_study,  'XGBoost')
]:
    trials = study.trials
    values = [t.value for t in trials if t.value is not None]
    best   = np.minimum.accumulate(values)

    ax.plot(values, alpha=0.4, color='steelblue', lw=0.8, label='Trial RMSE')
    ax.plot(best,   color='red',       lw=1.5,          label='Best so far')
    ax.axhline(1_180_664, color='gray', ls='--', lw=1, label='Baseline')
    ax.set_title(f'{name} — Optuna history')
    ax.set_xlabel('Trial')
    ax.set_ylabel('OOF RMSE')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── Parameter importance (LGBM)
try:
    importances = optuna.importance.get_param_importances(lgbm_study)
    print('LGBM parameter importance (ảnh hưởng đến RMSE):')
    for param, imp in sorted(importances.items(), key=lambda x: -x[1]):
        bar = '█' * int(imp * 40)
        print(f'  {param:<22}: {bar} {imp:.3f}')
except Exception:
    pass

## 6 — Retrain với best params + optimal blend

In [ ]:
# ── Build final param dicts từ Optuna best
LGBM_BEST = {
    'objective'    : 'regression',
    'metric'       : 'rmse',
    'boosting_type': 'gbdt',
    'verbose'      : -1,
    'random_state' : RANDOM_SEED,
    'n_jobs'       : -1,
    'n_estimators' : 2000,
    **lgbm_study.best_params
}

XGB_BEST = {
    'objective'   : 'reg:squarederror',
    'eval_metric' : 'rmse',
    'booster'     : 'gbtree',
    'random_state': RANDOM_SEED,
    'n_jobs'      : -1,
    'verbosity'   : 0,
    'n_estimators': 2000,
    **xgb_study.best_params
}

# ── OOF predictions với best params để tìm optimal blend
print('Running OOF với best params để tìm blend weight...')

lgbm_oof = np.full(len(X), np.nan)
xgb_oof  = np.full(len(X), np.nan)
lgbm_best_iters = []
xgb_best_iters  = []

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_vl, y_vl = X[val_idx],   y[val_idx]

    # LGBM
    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    dval   = lgb.Dataset(X_vl, label=y_vl, reference=dtrain, free_raw_data=False)
    m_lgbm = lgb.train(
        LGBM_BEST, dtrain,
        num_boost_round = 2000,
        valid_sets      = [dval],
        callbacks       = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)]
    )
    lgbm_oof[val_idx] = np.expm1(m_lgbm.predict(X_vl, num_iteration=m_lgbm.best_iteration))
    lgbm_best_iters.append(m_lgbm.best_iteration)

    # XGB
    m_xgb = xgb.XGBRegressor(**XGB_BEST, early_stopping_rounds=100)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
    xgb_oof[val_idx] = np.expm1(m_xgb.predict(X_vl))
    xgb_best_iters.append(m_xgb.best_iteration)

    print(f'  Fold {fold_idx+1}: lgbm_iter={lgbm_best_iters[-1]}  xgb_iter={xgb_best_iters[-1]}')

# ── Optimal blend weight
oof_mask   = ~np.isnan(lgbm_oof) & ~np.isnan(xgb_oof)
oof_target = train_df.loc[oof_mask, TARGET].values
lgbm_clean = np.clip(lgbm_oof[oof_mask], 0, None)
xgb_clean  = np.clip(xgb_oof[oof_mask],  0, None)

def blend_rmse(w):
    return np.sqrt(mean_squared_error(oof_target, w * lgbm_clean + (1-w) * xgb_clean))

result   = minimize_scalar(blend_rmse, bounds=(0.0, 1.0), method='bounded')
BLEND_W  = result.x
oof_blend = BLEND_W * lgbm_clean + (1-BLEND_W) * xgb_clean

# ── Metrics
def metrics(y_true, y_pred):
    return {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae' : mean_absolute_error(y_true, y_pred),
        'r2'  : r2_score(y_true, y_pred),
    }

m_lgbm = metrics(oof_target, lgbm_clean)
m_xgb  = metrics(oof_target, xgb_clean)
m_ens  = metrics(oof_target, oof_blend)

print()
print(f'Optimal blend: LGBM={BLEND_W:.4f}, XGB={1-BLEND_W:.4f}')
print()
print(f'{"":<12} {"RMSE":>12} {"MAE":>12} {"R²":>8}')
print('-' * 48)
print(f'{"Baseline":<12} {1_180_664:>12,.0f} {850_249:>12,.0f} {0.8170:>8.4f}  ← v2 ensemble')
print(f'{"LGBM tuned":<12} {m_lgbm["rmse"]:>12,.0f} {m_lgbm["mae"]:>12,.0f} {m_lgbm["r2"]:>8.4f}')
print(f'{"XGB tuned":<12} {m_xgb["rmse"]:>12,.0f} {m_xgb["mae"]:>12,.0f} {m_xgb["r2"]:>8.4f}')
print(f'{"Ensemble":<12} {m_ens["rmse"]:>12,.0f} {m_ens["mae"]:>12,.0f} {m_ens["r2"]:>8.4f}  ← NEW')
print()
print(f'RMSE improvement: {1_180_664 - m_ens["rmse"]:+,.0f} ({(1_180_664 - m_ens["rmse"])/1_180_664*100:+.1f}%)')
print(f'MAE  improvement: {850_249  - m_ens["mae"]:+,.0f} ({(850_249 - m_ens["mae"])/850_249*100:+.1f}%)')

## 7 — Final model + submission

In [ ]:
# n_estimators = mean(best_iters) + 10% buffer
lgbm_final_n = int(np.mean(lgbm_best_iters) * 1.1)
xgb_final_n  = int(np.mean(xgb_best_iters)  * 1.1)

print(f'LGBM final n_estimators: {lgbm_final_n}')
print(f'XGB  final n_estimators: {xgb_final_n}')
print('Retraining on full dataset...')

# LGBM
dtrain_full = lgb.Dataset(X, label=y, feature_name=FEATURE_COLS)
final_lgbm  = lgb.train(
    {**LGBM_BEST, 'n_estimators': lgbm_final_n},
    dtrain_full,
    num_boost_round = lgbm_final_n,
    callbacks       = [lgb.log_evaluation(500)]
)

# XGB
final_xgb = xgb.XGBRegressor(**{**XGB_BEST, 'n_estimators': xgb_final_n})
final_xgb.fit(X, y, verbose=False)

print('✅ Final models trained')

# Predict
X_pred    = pred_df[FEATURE_COLS].values
lgbm_test = np.expm1(final_lgbm.predict(X_pred))
xgb_test  = np.expm1(final_xgb.predict(X_pred))

pred_revenue = np.clip(BLEND_W * lgbm_test + (1-BLEND_W) * xgb_test, 0, None)
pred_cogs    = pred_revenue / COGS_RATIO

submission = pd.DataFrame({
    'Date'   : pred_df[DATE_COL].dt.strftime('%Y-%m-%d'),
    'Revenue': np.round(pred_revenue, 2),
    'COGS'   : np.round(pred_cogs, 2),
})
submission.to_csv(OUTPUT_FILE, index=False)

print(f'\n✅ Saved → {OUTPUT_FILE}')
print(f'Revenue: mean={pred_revenue.mean():,.0f}  min={pred_revenue.min():,.0f}  max={pred_revenue.max():,.0f}')

# Save best params for reference
best_params_out = {
    'lgbm_best_params' : lgbm_study.best_params,
    'xgb_best_params'  : xgb_study.best_params,
    'blend_weight_lgbm': BLEND_W,
    'oof_rmse'         : m_ens['rmse'],
    'oof_mae'          : m_ens['mae'],
    'oof_r2'           : m_ens['r2'],
}
with open(WORK_DIR + 'best_params.json', 'w') as f:
    json.dump(best_params_out, f, indent=2)

print('✅ Best params saved → /kaggle/working/best_params.json')

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_df[DATE_COL], train_df[TARGET], lw=0.7, label='Historical', color='steelblue')
ax.plot(pred_df[DATE_COL],  pred_revenue,     lw=0.9, label='Predicted',  color='salmon')
ax.axvline(train_df[DATE_COL].max(), color='gray', ls='--', lw=0.8)
ax.set_title(f'Revenue Forecast — Optuna Tuned | RMSE={m_ens["rmse"]:,.0f} | MAE={m_ens["mae"]:,.0f} | R²={m_ens["r2"]:.4f}')
ax.legend()
plt.tight_layout()
plt.show()